### **6. Social Determinants of Health**

- How does socioeconomic status (income and education) relate to the prevalence of various chronic diseases?

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
from utils.data import Data
data = Data()

variables = ['CRGVPRB3', '_EDUCAG', '_INCOMG1']
raw_data = data.extract_data_by_columns(variables)

df = pd.DataFrame({
    'CD': raw_data['CRGVPRB3']['data'],        
    'edu': raw_data['_EDUCAG']['data'],       
    'income': raw_data['_INCOMG1']['data'],   
}).apply(pd.to_numeric, errors='coerce')


chronic_diseases = [1, 2, 3, 4, 7, 8] 
df = df[
    df['CD'].isin(chronic_diseases) &
    (df['edu'] != 9) &
    (df['income'] != 9)
]

X = df[['edu', 'income']]
y = df['CD']

encoder = OneHotEncoder(drop='first', sparse_output=False)  
X_encoded = encoder.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

multi_logit_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500)
multi_logit_model.fit(X_train, y_train)

y_pred = multi_logit_model.predict(X_test)
print(classification_report(y_test, y_pred))

feature_names = encoder.get_feature_names_out(['edu', 'income'])
coefficients = multi_logit_model.coef_

print("Feature Weights for Each Class:")
for i, class_label in enumerate(multi_logit_model.classes_):
    print(f"Class {class_label} ({class_label}):")
    for feature, weight in zip(feature_names, coefficients[i]):
        print(f"  {feature}: {weight:.4f}")

2024-12-15 14:07:10,543 - INFO - Extracted 349 metadata fields from the HTML file.


              precision    recall  f1-score   support

         1.0       0.00      0.00      0.00        67
         2.0       0.00      0.00      0.00         6
         3.0       0.28      0.53      0.37       134
         4.0       0.00      0.00      0.00        61
         7.0       0.00      0.00      0.00        63
         8.0       0.35      0.54      0.42       138

    accuracy                           0.31       469
   macro avg       0.10      0.18      0.13       469
weighted avg       0.18      0.31      0.23       469

Feature Weights for Each Class:
Class 1.0 (1.0):
  edu_2: 0.1213
  edu_3: 0.1033
  edu_4: 0.0100
  income_2: 0.2297
  income_3: 0.2378
  income_4: 0.1180
  income_5: 0.1617
  income_6: 0.1862
  income_7: 0.5234
Class 2.0 (2.0):
  edu_2: -0.1673
  edu_3: -0.1879
  edu_4: -0.1696
  income_2: 0.0954
  income_3: -0.3119
  income_4: -0.2768
  income_5: -0.3983
  income_6: -0.3242
  income_7: -0.1097
Class 3.0 (3.0):
  edu_2: -0.0288
  edu_3: 0.0494
  edu_4: 

/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/zi

- What are the impacts of health disparities based on race, ethnicity, and income level across states?

In [17]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import OneHotEncoder
from utils.data import Data

# Load data
data = Data()
variables = ['CVDCRHD4', '_RACEGR3', '_INCOMG1', '_STATE']
raw_data = data.extract_data_by_columns(variables)

# Create DataFrame
df = pd.DataFrame({
    'stroke': raw_data['CVDCRHD4']['data'],  # 1: Yes, 2: No
    'Race/Ethnicity': raw_data['_RACEGR3']['data'],  
    'Income': raw_data['_INCOMG1']['data'],  
    '_STATE': raw_data['_STATE']['data']  # State FIPS Code
}).apply(pd.to_numeric, errors='coerce')

# Filter valid data
df = df[
    (df['stroke'].isin([1, 2])) &
    (df['Race/Ethnicity'].isin([1, 2, 3, 4, 5])) &  # Valid Race/Ethnicity values
    (df['Income'].isin([1, 2, 3, 4, 5, 6, 7]))        # Valid Income values
]

# Map labels for Race/Ethnicity and Income
race_labels = {
    1: 'White only, Non-Hispanic',
    2: 'Black only, Non-Hispanic',
    3: 'Other race only, Non-Hispanic',
    4: 'Multiracial, Non-Hispanic',
    5: 'Hispanic'
}
income_labels = {
    1: '<$15,000',
    2: '$15,000-$24,999',
    3: '$25,000-$34,999',
    4: '$35,000-$49,999',
    5: '$50,000-$99,999',
    6: '$100,000-$199,999',
    7: '$200,000 or more'
}

df['Race/Ethnicity'] = df['Race/Ethnicity'].map(race_labels)
df['Income'] = df['Income'].map(income_labels)

state_mapping_list = data.get_values_and_labels(['_STATE']).get('_STATE', [])
state_mapping = {item["Value"]: item["Label"] for item in state_mapping_list}
df['State Name'] = df['_STATE'].map(state_mapping)

# Logistic regression for each state
state_results = []
for state, state_data in df.groupby('_STATE'):

    # One-hot encode Race/Ethnicity and Income
    encoder_race = OneHotEncoder(drop='first', sparse_output=False)
    race_encoded = encoder_race.fit_transform(state_data[['Race/Ethnicity']])
    race_columns = encoder_race.get_feature_names_out(['Race/Ethnicity'])

    encoder_income = OneHotEncoder(drop='first', sparse_output=False)
    income_encoded = encoder_income.fit_transform(state_data[['Income']])
    income_columns = encoder_income.get_feature_names_out(['Income'])

    # Combine one-hot encoded variables
    encoded_df = pd.concat([
        pd.DataFrame(race_encoded, columns=race_columns, index=state_data.index),
        pd.DataFrame(income_encoded, columns=income_columns, index=state_data.index)
    ], axis=1)

    # Add constant term
    X = sm.add_constant(encoded_df)
    y = (state_data['stroke'] == 1).astype(int)

    # Skip state if data is insufficient for logistic regression
    if len(y.unique()) < 2 or len(y) < 50:
        continue

    # Logistic regression
    model = sm.Logit(y, X)
    result = model.fit(disp=False)

    # Extract p-values for each race and income group
    p_values = result.pvalues.to_dict()
    state_results.append({
        '_State': state,
        **{col: p_values.get(col, None) for col in race_columns},
        **{col: p_values.get(col, None) for col in income_columns}
    })

# Convert results to DataFrame
state_results_df = pd.DataFrame(state_results)

# Map state codes to names
state_results_df['State'] = state_results_df['_State'].map(state_mapping)

# Save results to CSV (optional)
state_results_df.to_csv('state.csv', index=False)

# Output results
print("State-Level Logistic Regression P-Values:")
print(state_results_df)

2024-12-15 14:07:13,152 - INFO - Extracted 349 metadata fields from the HTML file.
/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/zihanma/anaconda3/envs/islp/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_re

State-Level Logistic Regression P-Values:
    _State  Race/Ethnicity_Hispanic  Race/Ethnicity_Multiracial, Non-Hispanic  \
0        1                 0.577605                                  0.005489   
1        2                 0.579378                                  0.943111   
2        4                 0.000052                                  0.254000   
3        5                 0.009623                                  0.288128   
4        6                 0.000021                                  0.121088   
5        8                 0.042017                                  0.907709   
6        9                 0.553398                                  0.023410   
7       10                 0.048473                                  0.663684   
8       11                 0.314255                                  0.598134   
9       12                 0.150338                                  0.060672   
10      13                 0.002068                                